# Tendon vibration — P03 vs P04

**Only two participants have vibration recordings.** P05's 24-07 session is polarity x lidocaine;
its log does not mention vibration at all. So this is a two-participant comparison, not three.

## The thing that decides how this must be read

The vibrator was on a **different muscle and a different side** in each participant:

| | vibrator on | arm | conditions recorded |
|---|---|---|---|
| **P03** | wrist **extensor** | **left** | baseline, applied-but-off, ON (+ a 5 mA-step repeat for 30 Hz) |
| **P04** | **flexor** tendon | **right** | baseline, OFF, ON flexors, ON **extensors** (a check, vibrator moved) |

So P03's "VIB ON" and P04's "VIB ON extensors" are **not the same condition**. Comparing them
directly would compare left-extensor vibration against right-extensor vibration in different
people.

**What is comparable is the CHANGE at the vibrated site**, measured against that participant's own
baseline. Every bar below is a % of the same participant's baseline at the same intensity;
100 % = unchanged. The vibrated muscle is marked with `*`.

## One more caveat, from the logs

P03's "baseline" block was recorded at 16:31 with the **vibrator already attached** (applied
~15:45, switched off), and ~1.5 h into the session. P04's baseline is at 14:22, before the
vibrator was involved at all. So P03's baseline is closer to P04's *VIB OFF* than to P04's
baseline.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       vibration_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       check_thresholds, mt_stamp, warn_if_stale,
                       fig_threshold_grid,
                       fig_detection_row, fig_detection_grid)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D03 = "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/"
D04 = "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/"

RUN = {   # (subject, condition) -> csv.  Set PROTOCOL to "burst" or "arcex".
 "burst": {
   ("P03", "Baseline"):        D03 + "Burst_autosave_20260715_164431_737ms.csv",
   ("P03", "VIB off"):         D03 + "Burst_autosave_20260715_165831_999ms.csv",
   ("P03", "VIB ON"):          D03 + "Burst_autosave_20260715_170629_569ms.csv",
   ("P04", "Baseline"):        D04 + "Burst_autosave_20260716_142311_232ms.csv",
   ("P04", "VIB off"):         D04 + "Burst_autosave_20260716_144116_893ms.csv",
   ("P04", "VIB ON"):          D04 + "Burst_autosave_20260716_145100_665ms.csv",   # flexors
   ("P04", "VIB ON extensors"):D04 + "Burst_autosave_20260716_152821_805ms.csv",
 },
 "arcex": {
   ("P03", "Baseline"):        D03 + "Modulated_autosave_20260715_164630_593ms.csv",
   ("P03", "VIB off"):         D03 + "Modulated_autosave_20260715_170032_049ms.csv",
   ("P03", "VIB ON"):          D03 + "Modulated_autosave_20260715_171433_605ms.csv",
   ("P04", "Baseline"):        D04 + "Modulated_autosave_20260716_142536_785ms.csv",
   ("P04", "VIB off"):         D04 + "Modulated_autosave_20260716_144439_044ms.csv",
   ("P04", "VIB ON"):          D04 + "Modulated_autosave_20260716_145845_958ms.csv",  # flexors
   ("P04", "VIB ON extensors"):D04 + "Modulated_autosave_20260716_153556_059ms.csv",
 },
}
PROTOCOL   = "burst"          # <-- "burst" or "arcex"
SUBJECTS   = ["P03", "P04"]
CONDITIONS = ["Baseline", "VIB off", "VIB ON", "VIB ON extensors"]
VIBRATED   = {"P03": "Ext. digitorum (L)",    # the muscle the vibrator sat on
              "P04": "Flex. digitorum (R)"}
SITE       = {"P03": "left wrist extensor",   # ...in words, for the panel titles
              "P04": "right flexor tendon"}
# the same four muscles as the baseline comparison
MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN[PROTOCOL]
for (s, cond), f in runs.items():
    meta = load_run(f)[0]
    print(f"{s}  {cond:18s} {f.split('/')[-1][-22:]}  {[m['amp_ma'] for m in meta][0]}-"
          f"{[m['amp_ma'] for m in meta][-1]} mA")

## 2 · The motor thresholds you chose

Read straight from `results/<session>/mt_*.csv`, so this is what you actually picked — nothing
here can substitute a different value. One row per recording, one column per muscle: every
intensity stacked, **your threshold in red**, the mA in the corner. `not picked` = nothing saved
for that muscle in that recording; change it in section 5.

The cell below then assembles the thresholds the analysis uses. Every condition of a participant
is measured at the intensity that participant's **Baseline** defines, so vibration cannot move
the intensity as well as the response.

In [ ]:
# every recording of this comparison, so the picks can be compared between conditions too
CHECK = {f"{s} · {cond}": runs[(s, cond)] for s in SUBJECTS for cond in CONDITIONS
         if (s, cond) in runs}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title=f"the thresholds saved for each recording — {PROTOCOL}")

In [ ]:
# hand-picked thresholds win wherever they exist; detection fills in the rest. Pick them in
# notebooks/motor_thresholds/ and re-run this cell - the source is printed below.
MT = {s: thresholds_for(runs[(s, "Baseline")], MUSCLES, consecutive=2, **KW)
      for s in SUBJECTS}
for s in SUBJECTS:
    print(f"{s}  ({threshold_source(runs[(s, 'Baseline')])})  "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[s].items())))

MT_STAMP = mt_stamp(runs)      # so the figures can tell you if a pick is saved after this

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — **each at the threshold you picked in that
recording** (the red trace in section 2 — so this checks your own picks, block by block). The
10 pulses are re-aligned on their own onset, with the max (v) and min (^) the peak-to-peak is
made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

> Section 4 is different on purpose: there every condition is measured at the **Baseline**
> threshold, so vibration cannot move the intensity as well as the response. A muscle drops out
> of section 4 when that baseline mA was never delivered in the later block — P03's VIB blocks
> step 10 mA where its baseline steps 5, so 35 and 45 mA do not exist there.

### If a peak you can see is not being taken

`TRY` changes the settings **for these figures only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

# same grid as section 2 - a row per recording, a column per muscle - but showing how the
# peaks are found at the threshold picked in THAT recording
fig_detection_grid(CHECK, picked, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=f"peak detection at each recording's own threshold — {PROTOCOL}",
                   **TRY)

## 4 · The effect of vibration

Every condition as a % of that participant's own baseline — **100 % = unchanged**. `*` marks the
muscle the vibrator was on: that bar is the effect you are testing, the others show how much the
recording drifted between blocks on its own, which is the scale any effect has to beat.

Every condition of a muscle is read at **one intensity**, or the bars would compare vibration
with current. That intensity is the lowest one **at or above the baseline threshold that every
block of that participant delivered** — blocks were not always run on the same ladder (P03's VIB
blocks step 10 mA where its baseline steps 5), so the exact threshold is sometimes missing from a
block where the muscle responds perfectly well. Where that happens the reading moves up one step
and the cell says so; the `mA` column of the table is always the intensity actually used. Set
`match="exact"` to drop those muscles instead.

In [ ]:
# every condition of a participant is read at ONE intensity, set by that participant's BASELINE
# threshold and nudged up only where a block never delivered it (match="common")
warn_if_stale(runs, MT_STAMP, used=[(s, "Baseline") for s in SUBJECTS])

tab = vibration_table(runs, MT, MUSCLES, CONDITIONS, base="Baseline", match="common", **KW)
fig_vibration(tab, SUBJECTS, MUSCLES, CONDITIONS, vibrated=VIBRATED, site=SITE,
              ylim=(0, 200),   # a bar past the top is marked ^ with its real value above it
              title=f"Tendon vibration - {PROTOCOL}",
              save=None)

### The numbers, so they can be checked

In [ ]:
print(f"{'subject':8s}{'muscle':22s}{'mA':>5s}" + "".join(f"{c[:15]:>17s}" for c in CONDITIONS))
for s in SUBJECTS:
    for m in MUSCLES:
        row = [tab.get((s, c_, m), {}).get("pct", float("nan")) for c_ in CONDITIONS]
        if not any(v == v for v in row):
            continue
        amp = next((tab[(s, c_, m)]["amp"] for c_ in CONDITIONS if (s, c_, m) in tab), None)
        star = " *" if VIBRATED.get(s) == m else "  "
        print(f"{s:8s}{m + star:22s}{amp:5g}"
              + "".join(f"{(f'{v:.0f} %' if v == v else '-'):>17s}" for v in row))

### Before vs during vibration, one figure per muscle

The 1st pulse, the 2nd, and the mean of pulses 2–10, with the EMG underneath in matching
colours.

In [ ]:
# ======== settings ========
SHOW_MUSCLES = MUSCLES      # the same four as the baseline comparison
BARS_COND    = ["Baseline", "VIB off", "VIB ON"]     # which conditions to put side by side
COND_COLOUR  = {"Baseline": "#9A9A9A", "VIB off": "#C6C6C6", "VIB ON": "#C0392B",
                "VIB ON extensors": "#2E6F95"}
# ==========================

# the intensities the table above settled on, so these figures and that one cannot disagree
warn_if_stale(runs, MT_STAMP, used=[(s, "Baseline") for s in SUBJECTS])
MT_cond = {(s, cond): {m: tab[(s, cond, m)]["amp"] for m in SHOW_MUSCLES if (s, cond, m) in tab}
           for s in SUBJECTS for cond in BARS_COND if (s, cond) in runs}

for m in SHOW_MUSCLES:
    fig_bars(runs, MT_cond, m, SUBJECTS, protocols=tuple(BARS_COND), steps=0,
             colours=COND_COLOUR, names={c: c for c in BARS_COND},
             min_snr_ratio=1.5, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

## 5 · Change a threshold

One cell per recording: move the slider, press **Set to slider**, then **Save**. Re-run section 2
and every figure follows.

> Only a **Baseline** pick changes sections 3 and 4 — every condition is analysed at the baseline
> threshold. Re-picking a VIB block changes nothing but its own row in section 2.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not the baseline, and not
    whatever is already saved (that is the orange trace)."""
    csv = RUN[key] if isinstance(RUN, dict) and key in RUN else runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    print("   the analysis uses this participant's BASELINE thresholds: "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[key[0]].items())
                      if m in MUSCLES))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P03', 'Baseline'))

In [ ]:
repick(('P03', 'VIB off'))

In [ ]:
repick(('P03', 'VIB ON'))

In [ ]:
repick(('P04', 'Baseline'))

In [ ]:
repick(('P04', 'VIB off'))

In [ ]:
repick(('P04', 'VIB ON'))

In [ ]:
repick(('P04', 'VIB ON extensors'))